# RSNA Knee | DINOsaur V2 🦖 

It keeps the strongest proven components—multilingual graded report supervision, six clinically meaningful MRI slots, DINOv2-small, 2.5D three-slice inputs, target-specific slot attention and rank ensembling—while replacing the fragile parts of the pipeline.

Enabled by default:

- patient-space DICOM ordering with geometry → SliceLocation → InstanceNumber fallbacks;
- quality-aware sequence selection and localizer rejection;
- row/column-aware physical cropping and MONOCHROME1 correction;
- safe laterality normalization;
- fold-local monotonic calibration and reliability weighting of weak report labels;
- duplicate-report grouped, label-balanced folds;
- independent depth sampling per study;
- progressive DINOv2 unfreezing with layer-wise learning rates;
- two seeds × three folds, greedy checkpoint soups and dual-GPU DataParallel;
- gamma TTA and guarded target-specific depth pooling;
- immediate fallback `submission.csv` plus time-budget protection.

Attach the competition dataset and a local Hugging Face DINOv2-small checkpoint dataset. Internet must remain disabled. The notebook writes `submission.csv` and several diagnostic CSV files. A leaderboard improvement is plausible but cannot be guaranteed without executing against the hidden test set.

In [ ]:
from __future__ import annotations

import re
import unicodedata

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]


# Turkish dotted/dotless i must be folded before casefolding, otherwise "İZLENMEZ"
# and "izlenmez" diverge. ß and the Croatian/Serbian d-with-stroke likewise.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Fold case, diacritics and separators; keep Greek and Cyrillic letters.

    NFKD decomposition strips Latin accents and Greek tonos alike (ά -> α), which is what
    we want: reports are inconsistent about accents. It also maps the MICRO SIGN U+00B5
    to a real mu, which matters because most Greek reports here use the wrong codepoint.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")                    # soft hyphen
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Split into clauses, then attach `header:` lines to the value that follows.

    A report line reading `Fractures :` followed by `Aucune.` is one statement. Splitting
    on punctuation alone separates the anatomy from its negation and flips the label.
    """
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        # A fragment ending in a colon is a heading for the next fragment. Structured
        # English reports write long ones - "lateral compartment (meniscus, collateral
        # ligament complex, cartilage):" is eight words - so the cap is generous, and
        # the heading is kept as its own clause too in case it carries the finding.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        merged.append(c)
    # Comma-separated enumerations inside a long clause hide separate assertions.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


NEGATION = _rx(
    # en
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b",
    # es
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    # fr
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    # nl
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    # de
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    # tr
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    # hr / sr / bs
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    # el (accents already stripped)
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    # bg / ru
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

# Pathology vocabulary shared by the paired rules.
TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"\briss\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλιστ", r"дегенерат",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)


ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"предна кръстна", r"предната кръстна",
        # Plural, unqualified: reports routinely clear both cruciates in one clause
        # ("Ligamentos cruzados y colaterales dentro de limites normales"). Without
        # this, Spanish ACL was silent on 88% of its reports and Dutch on 70%.
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι",
        r"медиален колатерал", r"вътрешна странична",
        # Same plural pattern as the cruciates.
        # "Ligamentos cruzados y colaterales" separates the noun from its adjective, so
        # the adjective has to stand alone as a cue.
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# Osteoarthritis is rarely written as "osteoarthritis". It is written as cartilage loss,
# chondropathy grade, joint space narrowing, or osteophytes - scoped to a compartment.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# Self-declaring findings: the term itself is the finding.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        # "diz eklemi ici sivi miktari ... artmis" and "eklem icerisinde yaygin sivi
        # artisi" both occur; the noun takes a possessive suffix, so `eklem ` alone
        # misses. Match the stem plus any suffix.
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

# Terms that look like a finding but are not the finding being scored.
DECOY = {
    "Fracture": _rx(r"no fracture", r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}


STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\binterno\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bexterno\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")

# Fracture is the target whose stem varies most across the corpus.
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    # NOT a bare `fissur\w*`: "fisuras condrales" and "full thickness
                    # fissures in the articular cartilage" describe cartilage, not bone.
                    # The stem has to be anchored to a bone word to mean a fracture.
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True if a stem match has a qualifier within `window` characters either side.

    Character windows rather than token windows, because word order differs: English
    puts the side before the noun, Greek and Bulgarian often after, and Turkish
    attaches it as a separate preceding adjective.
    """
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False


# concept -> (stem, side) pairs used in addition to the phrase lexicons above
STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}


SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

# OA is often asserted for the whole joint rather than per compartment
# ("tricompartmental osteoarthritis", "gonarthrose", "incipient OA of all three
# compartments"). Those statements are evidence for all three OA targets.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

# A bare "bone marrow oedema" is not a contusion when it sits under a cartilage
# defect: subchondral oedema beneath a worn compartment is reactive degenerative signal,
# and reading it as a bruise turns every osteoarthritic knee into a trauma case.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


def _polarity(clause: str, anchor_end: int) -> str:
    """Classify one clause as positive, negative or uncertain for a matched term.

    Scope is the whole clause. Clause segmentation already keeps statements short, and
    a window in characters mis-scopes badly across languages with different word orders -
    Turkish puts its negator at the end of the sentence, English at the front.
    """
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        # "meniscus normal" negates; "normal ... but tear" does not.
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Phrase lexicon first, stem+side proximity as the fallback.

    Exposes `.search` so it drops into the same slot as a compiled pattern.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None


ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt]) for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause: str) -> float:
    """Weight one positive mention by how emphatic the sentence is.

    Ordered, not calibrated. A "moderate effusion" must outrank a "trace effusion" and
    both must outrank silence; the absolute numbers do not matter to AUC.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75                       # unqualified mention


def _is_structure_heading(clause: str) -> bool:
    """A short line ending in a colon names a section; it asserts nothing by itself.

    clauses() emits both the heading joined to the value beneath it and the heading on its
    own. For the five targets whose anatomy word *is* the finding - effusion, synovitis,
    Baker's, contusion, fracture - the bare heading matches the term with no negation in
    scope, so a structured report reading `Fractures :` / `Aucune.` scored a confident
    positive off the heading while the joined clause correctly read the negation. Short
    headings are therefore skipped as assertions; the joined clause carries the meaning.
    """
    return clause.endswith(":") and len(clause.split()) <= 5


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Accumulate graded evidence over clauses for one target.

    Returns (score, confidence, n_pos, n_neg). Positives are graded by severity and by
    optional context regexes; negatives only matter when nothing positive was found,
    because reports assert normality for every structure they check.
    """
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive" and _is_structure_heading(c):
            continue
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        # 0.52 .. 0.95, ordered by the strongest single mention, nudged by repetition.
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05          # silence sits above asserted-negative
    return score, conf, n_pos, n_neg


def extract(report: str) -> dict:
    """Extract twelve (score, confidence) pairs from one report."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            # Reactive subchondral oedema under a cartilage defect is osteoarthritis,
            # not a bruise. Explicit trauma wording pushes the other way.
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- cross-target corrections ------------------------------------------ #
    # A whole-joint osteoarthritis statement is evidence for every compartment that was
    # not separately assessed. Without this, "incipient OA of all three compartments"
    # scores zero on all three OA targets.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Synovitis is frequently visible on the images and absent from the text, so silence
    # is weak evidence of absence here in a way it is not for other findings. Effusion is
    # its most reliable textual proxy - the two share a mechanism - so a silent synovitis
    # inherits a fraction of the effusion evidence instead of falling to the floor.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out


# Additions are unioned onto the compiled patterns rather than edited into them, so the
# diff against the original lexicon stays readable and every entry can be reverted alone.

NEGATION = _rx(
    NEGATION.pattern,
    r"\bnone\b", r"\bnil\b",                       # "Effusion: none."
    r"not (identified|seen|visuali[sz]ed|demonstrated|detected|present|appreciated)",
    r"\bnegative\b",
    r"\bningun[ao]?\b", r"no se (identifica|aprecia|visualiza|reconoce)",
    r"\bnenhum\w*",
    r"niet zichtbaar", r"\bafwezigheid\b",
    r"izlenmemistir", r"gorulmemistir",
    # Rejected: r"\bnon\b". It negates "fracture non deplacee", which is a
    # non-displaced fracture - a positive finding with a qualifier, not an absence.
)

NORMALITY = _rx(
    NORMALITY.pattern,
    r"\bwnl\b", r"sans particularite", r"sin particularidades",
)

# A decoy skips the clause; a negation scores it. "no fracture" belongs to the second.
DECOY["Fracture"] = _rx(r"microfractur", r"\bfracture (risk|prophyla)")

_checks = {
    ("Fracture", "Fractures :\nAucune."): "lt",
    ("Fracture", "No fracture is identified."): "lt",
    ("Fracture", "Fractures :\nFracture non deplacee du plateau tibial."): "gt",
    ("Fracture", "Microfracture of the trochlea was performed."): "silent",
    ("Effusion", "Effusion:\nNone."): "lt",
    ("Effusion", "Joint effusion: nil."): "lt",
    ("Effusion", "Moderate joint effusion."): "gt",
    ("Fracture", "Fracturas :\nNinguna."): "lt",
    ("Baker's", "Quiste de Baker :\nPresente."): "gt",
}
for (tgt, text), want in _checks.items():
    v = extract(text)[tgt]
    ok = (v < 0.3) if want == "lt" else (v > 0.5) if want == "gt" else (0.25 < v < 0.32)
    assert ok, f"{want}: {tgt} scored {v:.2f} on {text!r}"
print("lexicon repairs verified on 9 regression cases")

# --- extractor self-test -------------------------------------------------------------
# Every claim the section above makes about the extractor is checked here on a synthetic
# report in the relevant language. Rerun after any lexicon edit: a rule that stops firing
# does not raise, it silently emits a negative.

def _s(text):
    return extract(text)

_cases = {
    "en_tear":     "Complete tear of the anterior cruciate ligament.",
    "en_intact":   "The anterior cruciate ligament is intact.",
    "es_group":    "Ligamentos cruzados y colaterales dentro de limites normales.",
    "fr_heading":  "Fractures :\nAucune.",
    "tr_effusion": "Eklem icerisinde yaygin sivi artisi mevcuttur.",
    "el_meniscus": "Ρηξη του εσω μηνισκου.",
    "eff_large":   "Large joint effusion.",
    "eff_trace":   "Trace joint effusion.",
    "eff_silent":  "The menisci are intact.",
    "global_oa":   "Tricompartmental osteoarthritis.",
    "degen_marrow": "Subchondral bone marrow oedema beneath a full thickness cartilage defect.",
    "true_bruise": "Bone bruise of the lateral femoral condyle after acute trauma.",
}
R = {k: _s(v) for k, v in _cases.items()}

assert R["en_tear"]["ACL"] > 0.5, R["en_tear"]["ACL"]
assert R["en_intact"]["ACL"] < 0.3, R["en_intact"]["ACL"]
# plural group statement clears both cruciates and both collaterals at once
assert R["es_group"]["ACL"] < 0.3 and R["es_group"]["MCL"] < 0.3, (
    R["es_group"]["ACL"], R["es_group"]["MCL"])
# heading on one line, value on the next: the merge in clauses() is what keeps this negative
assert R["fr_heading"]["Fracture"] < 0.3, R["fr_heading"]["Fracture"]
assert R["tr_effusion"]["Effusion"] > 0.5, R["tr_effusion"]["Effusion"]
assert R["el_meniscus"]["Medial Meniscus"] > 0.5, R["el_meniscus"]["Medial Meniscus"]
# severity is ordered, which is all AUC reads
assert R["eff_large"]["Effusion"] > R["eff_trace"]["Effusion"] > R["eff_silent"]["Effusion"], (
    R["eff_large"]["Effusion"], R["eff_trace"]["Effusion"], R["eff_silent"]["Effusion"])
# a whole-joint statement reaches every compartment
assert min(R["global_oa"][t] for t in ("Medial OA", "Lateral OA", "PF OA")) > 0.5
# reactive subchondral oedema must not read as a bruise as strongly as explicit trauma
assert R["degen_marrow"]["Contusion"] < R["true_bruise"]["Contusion"], (
    R["degen_marrow"]["Contusion"], R["true_bruise"]["Contusion"])

print("extractor self-test passed: negation, group statements, heading merge, Turkish and")
print("Greek morphology, severity ordering, global OA propagation, contusion decoy")

In [ ]:
from __future__ import annotations

import os
for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_name, "4")

import gc
import math
import random
import re
import time
import traceback
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

T0 = time.time()
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

IMG = 224
CROP_MM = 160.0
GROUP = 3
N_GROUP = 3
CACHE_SLICES = GROUP * N_GROUP
CACHE_BUDGET_GB = 17.0
HDR_THREADS = 20
PIX_THREADS = 12

N_FOLDS = 3
SEEDS = (2026, 3407)
EPOCHS = 10
WARMUP_EPOCHS = 1
BATCH_STUDIES = 8
EVAL_BATCH = 12
MAX_UNFREEZE = 6
WARM_UNFREEZE = 2
LR_BACKBONE = 9e-6
LR_HEAD = 9e-4
LAYER_DECAY = 0.78
WEIGHT_DECAY = 0.025
WARMUP_FRAC = 0.10
GRAD_CLIP = 1.0
TOP_CHECKPOINTS = 4
GOLD_WEIGHT = 4.0
DEPTH_MIN_GAIN = 0.0025
DEPTH_GOLD_GUARD = 0.03
TTA_GAMMAS = (0.90, 1.00, 1.10)
TIME_BUDGET = 8.30 * 3600
TIME_PAD = 18 * 60

AUG_ROT_DEG = 7.0
AUG_SCALE = 0.07
AUG_SHIFT = 0.045
AUG_INTENSITY = 0.10
AUG_GAMMA = 0.10
AUG_NOISE = 0.018

LAT_FALLBACK = "auto"
LAT_MIN_AGREEMENT = 0.85
LAT_MIN_OFFSET_MM = 5.0

SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", "fluid", True),
    ("COR_FLUID_FS", "Coronal", "fluid", True),
    ("AX_FLUID_FS", "Axial", "fluid", True),
    ("SAG_FLUID_NOFS", "Sagittal", "fluid", False),
    ("COR_T1", "Coronal", "t1", False),
    ("SAG_T1", "Sagittal", "t1", False),
]
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")
_LOCALIZER_RX = re.compile(
    r"locali[sz]er|scout|survey|calibration|three[ -]?plane|3[ -]?plane|"
    r"tri[ -]?plane|locator|pilot"
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


def log(message):
    print(f"[{time.time() - T0:7.1f}s] {message}", flush=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]
    for candidate in candidates:
        if (candidate / "train.csv").is_file() and (candidate / "test.csv").is_file():
            return candidate
    base = Path("/kaggle/input")
    if base.is_dir():
        for first in sorted(p for p in base.iterdir() if p.is_dir()):
            for candidate in [first] + sorted(p for p in first.iterdir() if p.is_dir()):
                if (candidate / "train.csv").is_file() and (candidate / "test_series").is_dir():
                    return candidate
    raise FileNotFoundError("RSNA competition dataset mount was not found")


def find_dinov2():
    base = Path("/kaggle/input")
    hits = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in {"train_series", "test_series"}]
            low = root.lower()
            if "config.json" in files and "dinov2" in low:
                hits.append(Path(root))
    small = [p for p in hits if "small" in str(p).lower() or "vits14" in str(p).lower()]
    return small[0] if small else (hits[0] if hits else None)


def safe_auc(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    ok = np.isfinite(y) & np.isfinite(p)
    y = y[ok].astype(int)
    p = p[ok]
    if len(y) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))


def macro_auc(y, p):
    values = [safe_auc(y[:, j], p[:, j]) for j in range(p.shape[1])]
    return float(np.nanmean(values)) if np.isfinite(values).any() else np.nan


def sigmoid_np(x):
    x = np.clip(np.asarray(x, dtype=np.float64), -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def logit_np(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-4, 1.0 - 1e-4)
    return np.log(p / (1.0 - p))


def rank_columns(p):
    return pd.DataFrame(p).rank(axis=0, method="average", pct=True).to_numpy(np.float64)


ROOT = find_root()
DINO_PATH = find_dinov2()
if DINO_PATH is None:
    raise FileNotFoundError(
        "Attach a Kaggle dataset containing a local Hugging Face DINOv2-small checkpoint"
    )

set_seed(SEEDS[0])
log(f"input root: {ROOT}")
log(f"DINOv2 checkpoint: {DINO_PATH}")
log(f"GPUs: {torch.cuda.device_count()} | folds={N_FOLDS} seeds={list(SEEDS)}")


In [ ]:
def build_report_arrays(train_df):
    start = time.time()
    extracted = pd.DataFrame([extract(text) for text in train_df["Report"].fillna("")])
    raw = extracted[TARGETS].to_numpy(np.float32)
    conf = extracted[[f"{t}__conf" for t in TARGETS]].to_numpy(np.float32)
    npos = extracted[[f"{t}__npos" for t in TARGETS]].to_numpy(np.float32)
    nneg = extracted[[f"{t}__nneg" for t in TARGETS]].to_numpy(np.float32)
    silence = ((npos == 0) & (nneg == 0)).mean(0)
    log(f"report labeler: {len(train_df)} studies in {time.time() - start:.1f}s")
    return raw, conf, silence


def build_gold_arrays(train_df):
    values = train_df[TARGETS].to_numpy(np.float32)
    mask = np.isfinite(values).all(1)
    return values, mask


def make_report_group_keys(train_df):
    keys = []
    for uid, report in zip(train_df["StudyInstanceUID"].astype(str), train_df["Report"].fillna("")):
        normalized = normalize(report).strip()
        if len(normalized) < 12:
            keys.append(f"uid::{uid}")
        else:
            keys.append(normalized)
    return np.asarray(keys, dtype=object)


def balanced_group_folds(group_keys, pseudo_y, gold_y, gold_mask, n_folds=N_FOLDS, seed=2026):
    n = len(group_keys)
    rng = np.random.default_rng(seed)
    gold_pos = np.where(gold_mask[:, None], np.nan_to_num(gold_y, nan=0.0), 0.0)
    gold_neg = np.where(gold_mask[:, None], 1.0 - np.nan_to_num(gold_y, nan=0.0), 0.0)
    features = np.concatenate([pseudo_y, 2.8 * gold_pos, 0.9 * gold_neg], axis=1).astype(np.float64)

    members = defaultdict(list)
    for i, key in enumerate(group_keys):
        members[str(key)].append(i)

    global_sum = features.sum(0)
    target_sum = global_sum / n_folds
    target_size = n / n_folds
    rarity_scale = 1.0 / np.sqrt(global_sum + 1.0)

    groups = []
    for key, idx in members.items():
        idx = np.asarray(idx, dtype=np.int64)
        vector = features[idx].sum(0)
        rarity = float(np.square(vector * rarity_scale).sum())
        groups.append((key, idx, vector, rarity, rng.random()))
    groups.sort(key=lambda item: (-item[3], -len(item[1]), item[4]))

    fold_size = np.zeros(n_folds, dtype=np.float64)
    fold_sum = np.zeros((n_folds, features.shape[1]), dtype=np.float64)
    assignment = np.full(n, -1, dtype=np.int64)

    def total_cost(candidate_fold, size, vector):
        sizes = fold_size.copy()
        sums = fold_sum.copy()
        sizes[candidate_fold] += size
        sums[candidate_fold] += vector
        size_cost = np.mean(np.square((sizes - target_size) / max(target_size, 1.0)))
        feature_cost = np.mean(np.square((sums - target_sum) / np.sqrt(target_sum + 1.0)))
        empty_penalty = 0.01 * np.sum(sizes == 0)
        return feature_cost + 0.35 * size_cost + empty_penalty

    for _, idx, vector, _, _ in groups:
        costs = np.asarray([total_cost(f, len(idx), vector) for f in range(n_folds)])
        best = np.flatnonzero(np.isclose(costs, costs.min(), rtol=0.0, atol=1e-12))
        fold = int(best[np.argmin(fold_size[best])])
        assignment[idx] = fold
        fold_size[fold] += len(idx)
        fold_sum[fold] += vector

    assert np.all(assignment >= 0)
    return assignment


def fit_monotonic_calibrators(raw, gold_y, gold_mask, fit_indices):
    fit_set = np.zeros(len(raw), dtype=bool)
    fit_set[np.asarray(fit_indices, dtype=np.int64)] = True
    usable = fit_set & gold_mask
    rows = []
    params = []

    for j, target in enumerate(TARGETS):
        x_raw = raw[usable, j]
        y = gold_y[usable, j].astype(int)
        auc = safe_auc(y, x_raw)
        slope, intercept = 1.0, 0.0
        fitted = False

        if len(y) >= 12 and y.sum() >= 3 and (1 - y).sum() >= 3 and np.isfinite(auc) and auc > 0.53:
            x = logit_np(x_raw).reshape(-1, 1)
            try:
                model = LogisticRegression(
                    C=0.12,
                    solver="lbfgs",
                    class_weight=None,
                    max_iter=2000,
                )
                model.fit(x, y)
                candidate_slope = float(model.coef_[0, 0])
                candidate_intercept = float(model.intercept_[0])
                if candidate_slope > 0:
                    slope = float(np.clip(candidate_slope, 0.35, 3.0))
                    intercept = float(np.clip(candidate_intercept, -2.25, 2.25))
                    fitted = True
            except Exception:
                pass

        if not np.isfinite(auc):
            reliability = 0.20
        else:
            reliability = float(np.clip((auc - 0.50) / 0.38, 0.18, 1.0))

        params.append((slope, intercept, reliability))
        rows.append({
            "target": target,
            "gold_auc": auc,
            "slope": slope,
            "intercept": intercept,
            "reliability": reliability,
            "fitted": fitted,
            "n_gold": int(len(y)),
            "n_pos": int(y.sum()) if len(y) else 0,
        })

    return params, pd.DataFrame(rows)


def build_fold_targets(raw, conf, gold_y, gold_mask, train_indices):
    params, table = fit_monotonic_calibrators(raw, gold_y, gold_mask, train_indices)
    y = np.empty_like(raw, dtype=np.float32)
    w = np.empty_like(raw, dtype=np.float32)

    for j, (slope, intercept, reliability) in enumerate(params):
        calibrated = sigmoid_np(slope * logit_np(raw[:, j]) + intercept)
        y[:, j] = (0.5 + reliability * (calibrated - 0.5)).astype(np.float32)
        confidence = 0.15 + 0.85 * np.clip(conf[:, j], 0.0, 1.0)
        w[:, j] = (confidence * (0.20 + 0.80 * reliability)).astype(np.float32)

    y[gold_mask] = gold_y[gold_mask]
    w[gold_mask] = GOLD_WEIGHT
    return y, w, table


def fold_diagnostics(folds, raw, gold_y, gold_mask):
    rows = []
    for fold in range(N_FOLDS):
        idx = folds == fold
        row = {"fold": fold, "n": int(idx.sum()), "gold": int((idx & gold_mask).sum())}
        for j, target in enumerate(TARGETS):
            row[f"{target}_pseudo_pos"] = float((raw[idx, j] > 0.5).mean())
            exact = idx & gold_mask
            row[f"{target}_gold_pos"] = float(gold_y[exact, j].mean()) if exact.any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
HDR_TAGS = [
    "SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
    "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
    "ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing",
    "Rows", "Columns", "SliceLocation", "InstanceNumber",
    "PhotometricInterpretation",
]


def _stringify_dicom_value(value):
    if value is None:
        return None
    if isinstance(value, (list, tuple)) or type(value).__name__ == "MultiValue":
        return "|".join(str(x) for x in value)
    return str(value)


def _parse_multi(value, expected=None):
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return None
    try:
        values = [float(x) for x in str(value).replace("\\", "|").split("|") if str(x).strip()]
    except Exception:
        return None
    if expected is not None and len(values) < expected:
        return None
    return values


def probe_series(item):
    split, study, series, path = item
    row = {
        "split": split,
        "StudyInstanceUID": study,
        "SeriesInstanceUID": series,
        "dir": path,
    }
    try:
        files = sorted(entry.name for entry in os.scandir(path) if entry.name.lower().endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        sample = files[len(files) // 2]
        ds = pydicom.dcmread(
            os.path.join(path, sample),
            stop_before_pixels=True,
            force=True,
        )
        for tag in HDR_TAGS:
            row[tag] = _stringify_dicom_value(getattr(ds, tag, None))
    except Exception as exc:
        row["err"] = str(exc)[:180]
    return row


def walk_series(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame()
    for study in os.scandir(base):
        if not study.is_dir():
            continue
        for series in os.scandir(study.path):
            if series.is_dir():
                items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe_series, items))
    return pd.DataFrame(rows)


def annotate_series(headers, public_series):
    df = headers.copy()
    cols = [
        "StudyInstanceUID", "SeriesInstanceUID", "Fluid_Sensitive",
        "Fat_Suppression", "Anatomical_Plane",
    ]
    available = [c for c in cols if c in public_series.columns]
    df = df.merge(public_series[available], on=["StudyInstanceUID", "SeriesInstanceUID"], how="left")

    description = (
        df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna("")
    ).str.lower().str.replace(_SEP, " ", regex=True)
    df["description"] = description
    df["is_localizer"] = description.str.contains(_LOCALIZER_RX, regex=True)

    options = df["ScanOptions"].fillna("").str.upper().str.split("|")
    option_fs = options.apply(lambda values: any(str(v).strip() in FATSAT_OPTS for v in values))
    public_fs = pd.to_numeric(df.get("Fat_Suppression", 0), errors="coerce").fillna(0).astype(int) > 0
    df["fatsat"] = description.str.contains(_FATSAT_RX, regex=True) | option_fs | public_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = description.str.contains(_T1_RX, regex=True)
    t2 = description.str.contains(_T2_RX, regex=True)
    pdw = description.str.contains(_PD_RX, regex=True)

    df["weight"] = np.where(
        t1 & ~t2 & ~pdw,
        "T1",
        np.where(
            t2 & ~pdw,
            "T2",
            np.where(
                pdw,
                "PD",
                np.where(
                    gre,
                    "GRE",
                    np.where(tr < 800, "T1", np.where(te > 60, "T2", np.where(tr >= 800, "PD", "UNK"))),
                ),
            ),
        ),
    )
    public_fluid = pd.to_numeric(df.get("Fluid_Sensitive", 0), errors="coerce").fillna(0).astype(int) > 0
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"]) | public_fluid
    df["plane"] = df.get("Anatomical_Plane", pd.Series(index=df.index, dtype=object))

    spacing = df["PixelSpacing"].apply(_parse_multi)
    df["px_row"] = spacing.apply(lambda x: x[0] if x and len(x) >= 2 else np.nan)
    df["px_col"] = spacing.apply(lambda x: x[1] if x and len(x) >= 2 else np.nan)
    df["rows"] = pd.to_numeric(df["Rows"], errors="coerce")
    df["cols"] = pd.to_numeric(df["Columns"], errors="coerce")
    df["n_slices"] = pd.to_numeric(df["n_slices"], errors="coerce").fillna(0)

    depth = np.log1p(np.minimum(df["n_slices"], 96.0))
    matrix = np.log1p(np.sqrt(df["rows"].fillna(0) * df["cols"].fillna(0)))
    pixel_area = df["px_row"] * df["px_col"]
    median_area = float(pixel_area.median()) if np.isfinite(pixel_area.median()) else 1.0
    resolution = -0.35 * np.log(np.clip(pixel_area.fillna(median_area), 0.05, 25.0))
    too_long = np.maximum(df["n_slices"] - 140.0, 0.0) / 70.0
    df["quality"] = depth + 0.35 * matrix + resolution - too_long - 8.0 * df["is_localizer"].astype(float)
    return df


def _tag_side(group):
    values = []
    for column in ("Laterality", "ImageLaterality"):
        if column in group.columns:
            values.extend(str(x).strip().upper() for x in group[column].dropna())
    values = [x[0] for x in values if x and x[0] in {"L", "R"}]
    return values[0] if values else None


def _position_side(group):
    xs = []
    if "ImagePositionPatient" not in group.columns:
        return None
    for value in group["ImagePositionPatient"].dropna():
        parsed = _parse_multi(value, expected=3)
        if parsed:
            xs.append(parsed[0])
    if not xs:
        return None
    x = float(np.median(xs))
    if abs(x) < LAT_MIN_OFFSET_MM:
        return None
    return "R" if x < 0 else "L"


def laterality_map(headers):
    tag, position = {}, {}
    for study, group in headers.groupby("StudyInstanceUID"):
        tag[study] = _tag_side(group)
        position[study] = _position_side(group)
    comparable = [study for study in tag if tag[study] and position[study]]
    agreement = (
        float(np.mean([tag[study] == position[study] for study in comparable]))
        if comparable else np.nan
    )
    if LAT_FALLBACK == "on":
        use_fallback = True
    elif LAT_FALLBACK == "off":
        use_fallback = False
    else:
        use_fallback = bool(comparable) and np.isfinite(agreement) and agreement >= LAT_MIN_AGREEMENT
    side = {study: tag[study] or (position[study] if use_fallback else None) for study in tag}
    coverage = float(np.mean([value is not None for value in side.values()])) if side else 0.0
    tag_coverage = float(np.mean([value is not None for value in tag.values()])) if tag else 0.0
    log(
        f"laterality tags={tag_coverage:.1%}, coordinate agreement={agreement:.1%} "
        f"on {len(comparable)}, fallback={use_fallback}, final={coverage:.1%}"
    )
    return side


def pick_slots(series_df):
    result = {}
    for study, group in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, kind, fat_suppression in SLOTS:
            base = group[group["plane"] == plane]
            if kind == "fluid":
                base = base[base["fluid"]]
            elif kind == "t1":
                base = base[base["weight"] == "T1"]
            base = base[base["fatsat"] == fat_suppression]

            clean = base[~base["is_localizer"]]
            candidates = clean if len(clean) else base
            if len(candidates) == 0 and kind == "t1":
                fallback = group[(group["plane"] == plane) & (~group["fatsat"]) & (~group["fluid"])]
                clean_fallback = fallback[~fallback["is_localizer"]]
                candidates = clean_fallback if len(clean_fallback) else fallback
            if len(candidates):
                record = candidates.sort_values(
                    ["quality", "n_slices"], ascending=[False, False]
                ).iloc[0].copy()
                record["slot_plane"] = plane
                chosen[name] = record
        result[study] = chosen
    return result


def _safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def _slice_geometry(path, plane):
    try:
        ds = pydicom.dcmread(
            path,
            stop_before_pixels=True,
            specific_tags=[
                "ImagePositionPatient", "ImageOrientationPatient", "SliceLocation",
                "InstanceNumber", "PhotometricInterpretation",
            ],
            force=True,
        )
        ipp = getattr(ds, "ImagePositionPatient", None)
        iop = getattr(ds, "ImageOrientationPatient", None)
        geometry = np.nan
        if ipp is not None and iop is not None and len(ipp) >= 3 and len(iop) >= 6:
            position = np.asarray([float(x) for x in ipp[:3]], dtype=np.float64)
            row = np.asarray([float(x) for x in iop[:3]], dtype=np.float64)
            col = np.asarray([float(x) for x in iop[3:6]], dtype=np.float64)
            normal = np.cross(row, col)
            axis = {"Sagittal": 0, "Coronal": 1, "Axial": 2}.get(plane, int(np.argmax(np.abs(normal))))
            if abs(normal[axis]) > 1e-6:
                if normal[axis] < 0:
                    normal = -normal
                geometry = float(np.dot(position, normal))
            else:
                geometry = float(position[axis])
        slice_location = _safe_float(getattr(ds, "SliceLocation", np.nan))
        instance = _safe_float(getattr(ds, "InstanceNumber", np.nan))
        photo = str(getattr(ds, "PhotometricInterpretation", ""))
        return geometry, slice_location, instance, photo
    except Exception:
        return np.nan, np.nan, np.nan, ""


def ordered_series_files(record):
    paths = [os.path.join(record["dir"], name) for name in record["files"]]
    plane = record.get("slot_plane", record.get("plane", None))
    rows = []
    for path in paths:
        geometry, slice_location, instance, photo = _slice_geometry(path, plane)
        if np.isfinite(geometry):
            key = (0, geometry, 0.0, os.path.basename(path))
        elif np.isfinite(slice_location):
            key = (1, slice_location, 0.0, os.path.basename(path))
        elif np.isfinite(instance):
            key = (2, instance, 0.0, os.path.basename(path))
        else:
            key = (3, 0.0, 0.0, os.path.basename(path))
        rows.append((key, path, photo))
    rows.sort(key=lambda item: item[0])
    return rows


def central_sample_indices(n, count):
    if n <= 1:
        return np.zeros(count, dtype=np.int64)
    lo = int(round(0.18 * (n - 1)))
    hi = int(round(0.82 * (n - 1)))
    values = np.linspace(lo, max(lo, hi), count)
    return np.clip(np.rint(values).astype(np.int64), 0, n - 1)


def read_slot(record, n_slices=CACHE_SLICES, out_size=IMG):
    ordered = ordered_series_files(record)
    if not ordered:
        return None
    indices = central_sample_indices(len(ordered), n_slices)
    planes = []

    for index in indices:
        _, path, header_photo = ordered[int(index)]
        try:
            ds = pydicom.dcmread(path, force=True)
            array = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
            intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
            array = array * slope + intercept
            photo = str(getattr(ds, "PhotometricInterpretation", header_photo)).upper()
            if photo == "MONOCHROME1":
                array = float(array.max() + array.min()) - array
            planes.append(array)
        except Exception:
            planes.append(None)

    valid = [plane for plane in planes if plane is not None and plane.ndim == 2]
    if not valid:
        return None
    shapes = defaultdict(int)
    for plane in valid:
        shapes[plane.shape] += 1
    target_shape = max(shapes, key=shapes.get)
    repaired = [
        plane if plane is not None and plane.shape == target_shape else np.zeros(target_shape, np.float32)
        for plane in planes
    ]
    volume = np.stack(repaired).astype(np.float32)

    row_spacing = float(record.get("px_row", np.nan))
    col_spacing = float(record.get("px_col", np.nan))
    height, width = target_shape
    crop_h = height
    crop_w = width
    if np.isfinite(row_spacing) and row_spacing > 0:
        crop_h = int(np.clip(round(CROP_MM / row_spacing), 16, height))
    if np.isfinite(col_spacing) and col_spacing > 0:
        crop_w = int(np.clip(round(CROP_MM / col_spacing), 16, width))
    y0 = max(0, (height - crop_h) // 2)
    x0 = max(0, (width - crop_w) // 2)
    volume = volume[:, y0:y0 + crop_h, x0:x0 + crop_w]

    low, high = np.percentile(volume, [1.0, 99.0])
    volume = np.clip((volume - low) / max(float(high - low), 1e-6), 0.0, 1.0)
    tensor = torch.from_numpy(np.ascontiguousarray(volume)).unsqueeze(0)
    tensor = F.interpolate(tensor, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (tensor.squeeze(0) * 255.0).round().clamp(0, 255).to(torch.uint8)


def normalize_laterality(images, plane, side):
    if side != "R":
        return images
    if plane in {"Coronal", "Axial"}:
        return torch.flip(images, dims=[-1])
    return torch.flip(images, dims=[0])


def estimate_cache_gb(n_studies):
    return n_studies * N_SLOT * CACHE_SLICES * IMG * IMG / 1024 ** 3


def build_cache(slot_map, side_map, tag):
    studies = sorted(slot_map)
    predicted_gb = estimate_cache_gb(len(studies))
    if predicted_gb > CACHE_BUDGET_GB:
        raise MemoryError(
            f"{tag} cache needs {predicted_gb:.1f} GB, above configured {CACHE_BUDGET_GB:.1f} GB"
        )
    study_index = {study: i for i, study in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), dtype=np.uint8)
    mask = np.zeros((len(studies), N_SLOT), dtype=np.float32)
    jobs = [
        (study, slot_index, plane, slot_map[study][name])
        for study in studies
        for slot_index, (name, plane, _, _) in enumerate(SLOTS)
        if name in slot_map[study]
    ]
    log(f"{tag}: cache={cache.shape}, {cache.nbytes / 1024 ** 3:.1f} GB, series={len(jobs)}")

    chunk = 256
    completed = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for start in range(0, len(jobs), chunk):
            block = jobs[start:start + chunk]
            records = [job[3] for job in block]
            images = pool.map(read_slot, records)
            for (study, slot_index, plane, _), image in zip(block, images):
                completed += 1
                if image is None:
                    continue
                cache[study_index[study], slot_index] = normalize_laterality(
                    image, plane, side_map.get(study)
                ).numpy()
                mask[study_index[study], slot_index] = 1.0
            if completed % 2048 < chunk:
                log(f"{tag}: decoded {completed}/{len(jobs)}")
    return studies, cache, mask


In [ ]:
class TargetSlotHead(nn.Module):
    def __init__(self, dim, n_slots=N_SLOT, n_targets=len(TARGETS), hidden=256, dropout=0.20):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_embedding = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_targets, hidden) * 0.02)
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden, n_targets)
        self.hidden = hidden

    def forward(self, features, mask):
        hidden = self.proj(features) + self.slot_embedding.unsqueeze(0)
        attention = torch.einsum("bsh,th->bts", hidden, self.query) / math.sqrt(self.hidden)
        attention = attention.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.dropout(torch.einsum("bts,bsh->bth", attention, hidden))
        return (context * self.output.weight.unsqueeze(0)).sum(-1) + self.output.bias


class KneeModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = TargetSlotHead(backbone.config.hidden_size * 2)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images, mask, gamma=1.0):
        batch, slots = images.shape[:2]
        x = images.reshape(batch * slots, *images.shape[2:]).float().div_(255.0)
        if gamma != 1.0:
            x = x.clamp(0.0, 1.0).pow(float(gamma))
        x = (x - self.mean) / self.std
        tokens = self.backbone(pixel_values=x).last_hidden_state
        features = torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], dim=1)
        return self.head(features.reshape(batch, slots, -1), mask)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def build_model(device):
    from transformers import AutoModel

    backbone = AutoModel.from_pretrained(str(DINO_PATH), local_files_only=True)
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    layers = backbone.encoder.layer
    for layer in layers[max(0, len(layers) - MAX_UNFREEZE):]:
        for parameter in layer.parameters():
            parameter.requires_grad = True
    for parameter in backbone.layernorm.parameters():
        parameter.requires_grad = True

    model = KneeModel(backbone).to(device)
    set_unfreeze_stage(model, warm=True)
    if device.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model


def set_unfreeze_stage(model, warm):
    core = unwrap_model(model)
    layers = core.backbone.encoder.layer
    start = len(layers) - (WARM_UNFREEZE if warm else MAX_UNFREEZE)
    for index, layer in enumerate(layers):
        enabled = index >= max(0, start)
        for parameter in layer.parameters():
            parameter.requires_grad = enabled
    for parameter in core.backbone.layernorm.parameters():
        parameter.requires_grad = True
    for parameter in core.head.parameters():
        parameter.requires_grad = True


def optimizer_groups(model):
    core = unwrap_model(model)
    layers = core.backbone.encoder.layer
    groups = []
    selected = list(range(max(0, len(layers) - MAX_UNFREEZE), len(layers)))
    for index in selected:
        distance = len(layers) - 1 - index
        learning_rate = LR_BACKBONE * (LAYER_DECAY ** distance)
        groups.append({"params": list(layers[index].parameters()), "lr": learning_rate})
    groups.append({"params": list(core.backbone.layernorm.parameters()), "lr": LR_BACKBONE})
    groups.append({"params": list(core.head.parameters()), "lr": LR_HEAD})
    return groups


def select_random_depth(cache_rows, group_ids):
    batch, slots, _, height, width = cache_rows.shape
    reshaped = cache_rows.view(batch, slots, N_GROUP, GROUP, height, width)
    batch_index = torch.arange(batch, device=cache_rows.device)
    return reshaped[batch_index, :, group_ids]


def select_fixed_depth(cache_rows, group_index):
    start = group_index * GROUP
    return cache_rows[:, :, start:start + GROUP]


def augment_bag(images):
    batch, slots, channels, height, width = images.shape
    x = images.float()
    angles = (torch.rand(batch, device=x.device) * 2.0 - 1.0) * math.radians(AUG_ROT_DEG)
    scales = 1.0 + (torch.rand(batch, device=x.device) * 2.0 - 1.0) * AUG_SCALE
    shifts_x = (torch.rand(batch, device=x.device) * 2.0 - 1.0) * AUG_SHIFT
    shifts_y = (torch.rand(batch, device=x.device) * 2.0 - 1.0) * AUG_SHIFT
    cosine = torch.cos(angles) / scales
    sine = torch.sin(angles) / scales
    theta = torch.zeros(batch, 2, 3, device=x.device, dtype=torch.float32)
    theta[:, 0, 0] = cosine
    theta[:, 0, 1] = -sine
    theta[:, 1, 0] = sine
    theta[:, 1, 1] = cosine
    theta[:, 0, 2] = shifts_x
    theta[:, 1, 2] = shifts_y
    theta = theta.repeat_interleave(slots, dim=0)

    flat = x.reshape(batch * slots, channels, height, width)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    flat = F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    x = flat.reshape(batch, slots, channels, height, width)

    intensity = 1.0 + (torch.rand(batch, 1, 1, 1, 1, device=x.device) * 2.0 - 1.0) * AUG_INTENSITY
    gamma = 1.0 + (torch.rand(batch, 1, 1, 1, 1, device=x.device) * 2.0 - 1.0) * AUG_GAMMA
    x = (x / 255.0).clamp(0.0, 1.0).pow(gamma) * 255.0
    x = x * intensity
    if AUG_NOISE > 0:
        noise = torch.randn_like(x) * (255.0 * AUG_NOISE)
        x = x + noise
    return x.round().clamp(0, 255).to(torch.uint8)


def cosine_schedule(step, total_steps):
    warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
    if step < warmup_steps:
        return float(step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))


@torch.no_grad()
def predict_depth_logits(model, cache, mask, indices, device, gammas=(1.0,)):
    model.eval()
    indices = np.asarray(indices, dtype=np.int64)
    result = np.zeros((len(indices), N_GROUP, len(TARGETS)), dtype=np.float32)
    for start in range(0, len(indices), EVAL_BATCH):
        selected = indices[start:start + EVAL_BATCH]
        rows = torch.from_numpy(cache[selected]).to(device, non_blocking=True)
        present = torch.from_numpy(mask[selected]).to(device, non_blocking=True)
        for group_index in range(N_GROUP):
            images = select_fixed_depth(rows, group_index)
            accumulated = None
            for gamma in gammas:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                    logits = model(images, present, gamma=float(gamma)).float()
                accumulated = logits if accumulated is None else accumulated + logits
            result[start:start + len(selected), group_index] = (
                accumulated / len(gammas)
            ).cpu().numpy()
    return result


def depth_prediction_candidates(depth_logits):
    probabilities = sigmoid_np(depth_logits)
    candidates = {
        "mean_logit": sigmoid_np(depth_logits.mean(1)),
        "mean_probability": probabilities.mean(1),
        "maximum_probability": probabilities.max(1),
    }
    if depth_logits.shape[1] >= 2:
        candidates["top_two_probability"] = np.sort(probabilities, axis=1)[:, -2:].mean(1)
    else:
        candidates["top_two_probability"] = probabilities.mean(1)
    return candidates


def apply_depth_rules(depth_logits, rules):
    candidates = depth_prediction_candidates(depth_logits)
    output = np.empty((len(depth_logits), len(TARGETS)), dtype=np.float64)
    for target_index, rule in enumerate(rules):
        output[:, target_index] = candidates[rule][:, target_index]
    return output


def selection_score(derived_auc, gold_auc):
    if not np.isfinite(derived_auc):
        return -np.inf
    if not np.isfinite(gold_auc):
        return float(derived_auc)
    return float(0.87 * derived_auc + 0.13 * gold_auc)


def clone_state(model):
    return {
        name: value.detach().cpu().clone()
        for name, value in unwrap_model(model).state_dict().items()
    }


def average_states(current, candidate, current_count):
    averaged = {}
    denominator = current_count + 1.0
    for name, value in current.items():
        other = candidate[name]
        if value.dtype.is_floating_point:
            averaged[name] = (value * current_count + other) / denominator
        else:
            averaged[name] = value.clone()
    return averaged


def checkpoint_metrics(model, cache, mask, val_indices, raw, gold_y, gold_mask, device):
    depth_logits = predict_depth_logits(model, cache, mask, val_indices, device, gammas=(1.0,))
    predictions = depth_prediction_candidates(depth_logits)["mean_logit"]
    derived_y = (raw[val_indices] > 0.5).astype(np.int8)
    derived_auc = macro_auc(derived_y, predictions)
    exact_local = gold_mask[val_indices]
    gold_auc = (
        macro_auc(gold_y[val_indices][exact_local].astype(np.int8), predictions[exact_local])
        if exact_local.sum() >= 8 else np.nan
    )
    return selection_score(derived_auc, gold_auc), derived_auc, gold_auc, depth_logits


def train_one_model(
    fold,
    seed,
    cache,
    mask,
    train_indices,
    val_indices,
    targets,
    weights,
    raw,
    gold_y,
    gold_mask,
    device,
):
    set_seed(seed)
    model = build_model(device)
    parameter_groups = optimizer_groups(model)
    optimizer = torch.optim.AdamW(parameter_groups, weight_decay=WEIGHT_DECAY)
    batches_per_epoch = max(1, math.ceil(len(train_indices) / BATCH_STUDIES))
    total_steps = max(1, EPOCHS * batches_per_epoch)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: cosine_schedule(step, total_steps)
    )
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    checkpoints = []
    history = []
    global_step = 0

    for epoch in range(EPOCHS):
        warm = epoch < WARMUP_EPOCHS
        set_unfreeze_stage(model, warm=warm)
        model.train()
        rng = np.random.default_rng(seed * 100 + epoch)
        order = rng.permutation(train_indices)
        losses = []

        for start in range(0, len(order), BATCH_STUDIES):
            selected = order[start:start + BATCH_STUDIES]
            if len(selected) < 2:
                continue
            rows = torch.from_numpy(cache[selected]).to(device, non_blocking=True)
            group_ids = torch.randint(0, N_GROUP, (len(selected),), device=device)
            images = augment_bag(select_random_depth(rows, group_ids))
            present = torch.from_numpy(mask[selected]).to(device, non_blocking=True)
            y = torch.from_numpy(targets[selected]).to(device, non_blocking=True)
            w = torch.from_numpy(weights[selected]).to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images, present)
                loss_matrix = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
                loss = (loss_matrix * w).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1
            losses.append(float(loss.detach().cpu()))

            if time.time() - T0 > TIME_BUDGET - TIME_PAD:
                break

        score, derived_auc, gold_auc, _ = checkpoint_metrics(
            model, cache, mask, val_indices, raw, gold_y, gold_mask, device
        )
        history.append({
            "fold": fold,
            "seed": seed,
            "epoch": epoch + 1,
            "loss": float(np.mean(losses)) if losses else np.nan,
            "derived_auc": derived_auc,
            "gold_auc": gold_auc,
            "selection": score,
            "warm_stage": warm,
        })
        log(
            f"fold={fold} seed={seed} epoch={epoch + 1}/{EPOCHS} "
            f"loss={history[-1]['loss']:.4f} derived={derived_auc:.4f} "
            f"gold={gold_auc:.4f} select={score:.4f}"
        )
        checkpoints.append({
            "score": score,
            "derived": derived_auc,
            "gold": gold_auc,
            "state": clone_state(model),
        })
        checkpoints = sorted(checkpoints, key=lambda item: item["score"], reverse=True)[:TOP_CHECKPOINTS]

        if time.time() - T0 > TIME_BUDGET - TIME_PAD:
            log("time guard reached during training")
            break

    if not checkpoints:
        checkpoints = [{"score": -np.inf, "derived": np.nan, "gold": np.nan, "state": clone_state(model)}]

    del optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    soup_state, soup_count = checkpoints[0]["state"], 1
    unwrap_model(model).load_state_dict(soup_state)
    current_score, current_derived, current_gold, _ = checkpoint_metrics(
        model, cache, mask, val_indices, raw, gold_y, gold_mask, device
    )

    for candidate in checkpoints[1:]:
        trial_state = average_states(soup_state, candidate["state"], soup_count)
        unwrap_model(model).load_state_dict(trial_state)
        trial_score, trial_derived, trial_gold, _ = checkpoint_metrics(
            model, cache, mask, val_indices, raw, gold_y, gold_mask, device
        )
        gold_ok = (
            not np.isfinite(current_gold)
            or not np.isfinite(trial_gold)
            or trial_gold >= current_gold - 0.025
        )
        if trial_score >= current_score - 1e-5 and gold_ok:
            soup_state = trial_state
            soup_count += 1
            current_score, current_derived, current_gold = trial_score, trial_derived, trial_gold
        else:
            unwrap_model(model).load_state_dict(soup_state)

    log(
        f"fold={fold} seed={seed} soup={soup_count} "
        f"derived={current_derived:.4f} gold={current_gold:.4f} select={current_score:.4f}"
    )
    del model, checkpoints
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return soup_state, history


def choose_depth_rules(oof_logits, oof_count, raw, gold_y, gold_mask):
    valid = oof_count > 0
    averaged = np.zeros_like(oof_logits, dtype=np.float64)
    averaged[valid] = oof_logits[valid] / oof_count[valid, None, None]
    candidates = depth_prediction_candidates(averaged)
    rules = []
    rows = []

    for target_index, target in enumerate(TARGETS):
        y = (raw[:, target_index] > 0.5).astype(np.int8)
        base_auc = safe_auc(y[valid], candidates["mean_logit"][valid, target_index])
        exact = valid & gold_mask
        base_gold = safe_auc(
            gold_y[exact, target_index], candidates["mean_logit"][exact, target_index]
        )
        selected = "mean_logit"
        selected_auc = base_auc
        selected_gold = base_gold

        for name in ("mean_probability", "top_two_probability", "maximum_probability"):
            auc = safe_auc(y[valid], candidates[name][valid, target_index])
            gold_auc = safe_auc(gold_y[exact, target_index], candidates[name][exact, target_index])
            gain_needed = DEPTH_MIN_GAIN if np.isfinite(base_gold) else max(0.005, DEPTH_MIN_GAIN)
            gold_ok = (
                not np.isfinite(base_gold)
                or not np.isfinite(gold_auc)
                or gold_auc >= base_gold - DEPTH_GOLD_GUARD
            )
            if np.isfinite(auc) and auc >= selected_auc + gain_needed and gold_ok:
                selected = name
                selected_auc = auc
                selected_gold = gold_auc

        rules.append(selected)
        rows.append({
            "target": target,
            "rule": selected,
            "base_auc": base_auc,
            "selected_auc": selected_auc,
            "base_gold_auc": base_gold,
            "selected_gold_auc": selected_gold,
        })
    return rules, pd.DataFrame(rows), averaged


def write_benchmark_submission(test_df):
    submission = test_df[["StudyInstanceUID"]].copy()
    for target in TARGETS:
        submission[target] = 0.5
    submission.to_csv("submission.csv", index=False)


def write_rank_submission(model_predictions, cached_test_studies, test_df):
    if not model_predictions:
        return None
    rank_mean = np.mean([rank_columns(prediction) for prediction in model_predictions], axis=0)
    frame = pd.DataFrame(rank_mean, columns=TARGETS)
    frame.insert(0, "StudyInstanceUID", cached_test_studies)
    submission = test_df[["StudyInstanceUID"]].merge(frame, on="StudyInstanceUID", how="left")
    submission[TARGETS] = submission[TARGETS].fillna(0.5)
    submission.to_csv("submission.csv", index=False)
    return submission


In [ ]:
def main():
    train_df = pd.read_csv(ROOT / "train.csv")
    test_df = pd.read_csv(ROOT / "test.csv")
    train_series = pd.read_csv(ROOT / "train_series.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    write_benchmark_submission(test_df)
    log(f"train={train_df.shape} test={test_df.shape}")

    raw_full, conf_full, silence = build_report_arrays(train_df)
    gold_full, gold_mask_full = build_gold_arrays(train_df)
    group_keys_full = make_report_group_keys(train_df)

    log("scanning test series headers")
    test_headers = annotate_series(walk_series("test_series"), test_series)
    log(f"test series={len(test_headers)}")
    log("scanning train series headers")
    train_headers = annotate_series(walk_series("train_series"), train_series)
    log(f"train series={len(train_headers)}")

    train_side = laterality_map(train_headers)
    test_side = laterality_map(test_headers)
    train_slots = pick_slots(train_headers)
    test_slots = pick_slots(test_headers)

    coverage = pd.DataFrame({
        name: [float(np.mean([name in slots for slots in train_slots.values()]))]
        for name, _, _, _ in SLOTS
    })
    coverage.to_csv("slot_coverage.csv", index=False)
    log(f"mean selected slots={np.mean([len(value) for value in train_slots.values()]):.2f}")

    train_studies, train_cache, train_mask = build_cache(train_slots, train_side, "train")
    test_studies, test_cache, test_mask = build_cache(test_slots, test_side, "test")

    uid_to_row = {
        uid: index for index, uid in enumerate(train_df["StudyInstanceUID"].astype(str).tolist())
    }
    train_rows = np.asarray([uid_to_row[str(uid)] for uid in train_studies], dtype=np.int64)
    raw = raw_full[train_rows]
    conf = conf_full[train_rows]
    gold_y = gold_full[train_rows]
    gold_mask = gold_mask_full[train_rows]
    group_keys = group_keys_full[train_rows]

    balance_y = raw.copy()
    balance_y[gold_mask] = gold_y[gold_mask]
    folds = balanced_group_folds(group_keys, balance_y, gold_y, gold_mask, N_FOLDS, SEEDS[0])
    fold_table = fold_diagnostics(folds, raw, gold_y, gold_mask)
    fold_table.to_csv("fold_diagnostics.csv", index=False)
    log(
        "fold sizes=" + str([int((folds == fold).sum()) for fold in range(N_FOLDS)])
        + " gold=" + str([int(((folds == fold) & gold_mask).sum()) for fold in range(N_FOLDS)])
    )

    label_quality_rows = []
    for target_index, target in enumerate(TARGETS):
        auc = safe_auc(gold_y[gold_mask, target_index], raw[gold_mask, target_index])
        label_quality_rows.append({
            "target": target,
            "report_auc": auc,
            "silence_rate": float(silence[target_index]),
            "gold_positive": int(gold_y[gold_mask, target_index].sum()),
        })
    pd.DataFrame(label_quality_rows).to_csv("report_label_quality.csv", index=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    oof_logits_sum = np.zeros(
        (len(train_studies), N_GROUP, len(TARGETS)), dtype=np.float64
    )
    oof_count = np.zeros(len(train_studies), dtype=np.int32)
    test_depth_models = []
    provisional_predictions = []
    all_history = []
    calibration_tables = []

    for fold in range(N_FOLDS):
        train_indices = np.flatnonzero(folds != fold)
        val_indices = np.flatnonzero(folds == fold)
        targets, weights, calibration = build_fold_targets(
            raw, conf, gold_y, gold_mask, train_indices
        )
        calibration.insert(0, "fold", fold)
        calibration_tables.append(calibration)

        for seed in SEEDS:
            if time.time() - T0 > TIME_BUDGET - TIME_PAD:
                log("time guard prevents starting another model")
                break

            log(
                f"training fold={fold} seed={seed} train={len(train_indices)} "
                f"val={len(val_indices)} exact_val={int(gold_mask[val_indices].sum())}"
            )
            state, history = train_one_model(
                fold,
                seed,
                train_cache,
                train_mask,
                train_indices,
                val_indices,
                targets,
                weights,
                raw,
                gold_y,
                gold_mask,
                device,
            )
            all_history.extend(history)

            inference_model = build_model(device)
            unwrap_model(inference_model).load_state_dict(state)
            val_depth = predict_depth_logits(
                inference_model,
                train_cache,
                train_mask,
                val_indices,
                device,
                gammas=(1.0,),
            )
            test_depth = predict_depth_logits(
                inference_model,
                test_cache,
                test_mask,
                np.arange(len(test_studies)),
                device,
                gammas=TTA_GAMMAS,
            )
            oof_logits_sum[val_indices] += val_depth
            oof_count[val_indices] += 1
            test_depth_models.append(test_depth)

            default_prediction = depth_prediction_candidates(test_depth)["mean_logit"]
            provisional_predictions.append(default_prediction)
            submission = write_rank_submission(
                provisional_predictions, test_studies, test_df
            )
            log(
                f"finished models={len(test_depth_models)}; provisional submission={submission.shape}"
            )

            del inference_model, state, val_depth, test_depth, default_prediction
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if time.time() - T0 > TIME_BUDGET - TIME_PAD:
            break

    if calibration_tables:
        pd.concat(calibration_tables, ignore_index=True).to_csv(
            "weak_label_calibration.csv", index=False
        )
    if all_history:
        pd.DataFrame(all_history).to_csv("training_history.csv", index=False)

    if not test_depth_models:
        log("no model completed; benchmark submission remains")
        return None

    rules, depth_table, averaged_oof = choose_depth_rules(
        oof_logits_sum, oof_count, raw, gold_y, gold_mask
    )
    depth_table.to_csv("depth_pooling_rules.csv", index=False)
    final_predictions = [apply_depth_rules(depth_logits, rules) for depth_logits in test_depth_models]
    submission = write_rank_submission(final_predictions, test_studies, test_df)

    valid_oof = oof_count > 0
    oof_prediction = apply_depth_rules(averaged_oof[valid_oof], rules)
    pseudo_oof_auc = macro_auc((raw[valid_oof] > 0.5).astype(np.int8), oof_prediction)
    exact = valid_oof & gold_mask
    exact_oof_auc = (
        macro_auc(gold_y[exact].astype(np.int8), apply_depth_rules(averaged_oof[exact], rules))
        if exact.sum() >= 8 else np.nan
    )
    log(f"OOF pseudo macro AUC={pseudo_oof_auc:.4f}; exact macro AUC={exact_oof_auc:.4f}")
    log(f"depth rules={dict(zip(TARGETS, rules))}")
    log(f"final submission.csv={submission.shape}; models={len(test_depth_models)}")
    print(submission.head().to_string())
    return submission


try:
    FINAL_SUBMISSION = main()
except Exception:
    traceback.print_exc()
    try:
        test_frame = pd.read_csv(find_root() / "test.csv")
        write_benchmark_submission(test_frame)
        print("wrote fallback submission.csv")
    except Exception:
        traceback.print_exc()
log("done")
